# Tissue-Specific Simulations

This notebook demonstrates how to simulate specific real tissues using PointillSim.

We implement the tissue descriptions from `tissue_descriptions/` folder:
1. **Colon Crypts**: Glandular structures with stem-to-differentiated gradients
2. **Cortex Layers**: Layered neuronal tissue
3. **Mammary Gland**: Acinar structures with bilayer epithelium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.collections import PatchCollection

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    # Rules
    RandomCellTypeRule,
    SingleTypeRule,
    MixOfNCellTypesRule,
    ProbabilityNodeFieldRule,
    DistanceBasedRule,
)
from pointillsim.rules.composite import LayerRule, CompositeRule

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
# Helper function to visualize results
def visualize_tissue(fov, title, tissue=None, hybiss=None, figsize=(16, 8)):
    """Visualize a tissue simulation."""
    fig, axes = plt.subplots(1, 3 if hybiss else 2, figsize=figsize)
    
    # Cell types
    ax = axes[0]
    scatter = ax.scatter(
        fov.cell_centroids[:, 0],
        fov.cell_centroids[:, 1],
        c=fov.class_instance,
        cmap='Set1',
        s=15, alpha=0.7
    )
    ax.set_aspect('equal')
    ax.set_title(f'{title}\nCell Types ({len(fov.cell_centroids)} cells)')
    plt.colorbar(scatter, ax=ax, label='Cell Type')
    
    # Cell morphology (if available)
    ax = axes[1]
    if hasattr(fov, 'cell_colors') and fov.cell_colors is not None:
        ellipses = []
        colors = []
        for i in range(len(fov.cell_centroids)):
            ellipse = Ellipse(
                xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
                width=2 * fov.cell_major_axis[i],
                height=2 * fov.cell_minor_axis[i],
                angle=np.degrees(fov.cell_rotation[i]),
            )
            ellipses.append(ellipse)
            colors.append(fov.cell_colors[i])
        collection = PatchCollection(ellipses, alpha=0.6)
        collection.set_facecolors(colors)
        collection.set_edgecolors('black')
        collection.set_linewidths(0.3)
        ax.add_collection(collection)
        ax.autoscale()
    else:
        ax.scatter(
            fov.cell_centroids[:, 0],
            fov.cell_centroids[:, 1],
            c=fov.class_instance,
            cmap='Set1',
            s=30, alpha=0.7
        )
    ax.set_aspect('equal')
    ax.set_title('Cell Morphology')
    
    # Transcript dots (if available)
    if hybiss:
        ax = axes[2]
        dots_df = hybiss.make_pandas_df()
        ax.scatter(dots_df['x'], dots_df['y'], s=1, alpha=0.3, c='red')
        ax.set_aspect('equal')
        ax.set_title(f'Transcript Dots ({len(dots_df):,})')
    
    plt.tight_layout()
    plt.show()
    
    # Print cell type counts
    from collections import Counter
    counts = Counter(fov.class_instance)
    print("Cell type distribution:")
    for t in sorted(counts.keys()):
        print(f"  Type {t}: {counts[t]:4d} cells ({100*counts[t]/len(fov.class_instance):.1f}%)")

---
## 1. Colon Crypts

Simulating colonic epithelium with crypt structures:
- Background: Lamina propria (stroma + immune cells)
- Foreground: Multiple crypt structures with stem-to-differentiated gradients

In [ ]:
# Cell type indices
# 0: Stem cells, 1: Transit-amplifying, 2: Enterocytes, 3: Goblet cells
# 4: Stromal fibroblasts, 5: Immune cells

n_cell_types_colon = 6
frame_size_colon = 1200

# Create tissue expression profiles
tissue_colon = TissueCellTypes()
tissue_colon.generate_types_and_markers(n_genes=50, n_cell_types=n_cell_types_colon)
tissue_colon._cell_type_names = [
    "Stem", "Transit-amplifying", "Enterocyte", "Goblet", "Fibroblast", "Immune"
]

# Cell properties
cell_props_colon = CellTypesProperties(
    n_cell_types=n_cell_types_colon,
    sizes=[10, 11, 13, 14, 15, 10],  # Varying cell sizes
)

# Background: Lamina propria (stroma)
def colon_background():
    return FrameWideElement(
        frame_size=frame_size_colon,
        tipical_cell_spacing=28,  # Sparse stroma
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_colon,
            list_N=[4, 5],  # Fibroblasts + Immune
            proportions=[0.75, 0.25]
        )
    )

# Foreground: Crypts with stem-to-differentiated gradient
def colon_crypt():
    # DistanceBasedRule: stem at center, differentiated at edge
    # We'll use LayerRule for 3 zones: stem -> transit -> differentiated
    rule = LayerRule(
        n_cell_types=n_cell_types_colon,
        layer_types=[0, 1, 2],  # Stem, Transit-amp, Enterocyte
        layer_boundaries=[0.35, 0.7],  # Inner 35% stem, 35-70% transit, outer differentiated
        transition_width=8,
    )
    return VacuolatedStructure(
        frame_size=frame_size_colon,
        scale=45 + np.random.uniform(-10, 10),
        hole_scale_factor=0.55,  # Lumen in center
        rules=rule,
        tipical_cell_spacing=12,
    )

# Create FOV distribution
fov_dist_colon = FOVDistribution(
    frame_size=frame_size_colon,
    background_element=colon_background,
    other_elements=[colon_crypt],
    elements_frequency=[1.0],
    attempts_at_elements=18,  # ~18 crypts
)

# Generate
fov_colon = fov_dist_colon.generate_fov()
cell_props_colon.apply(fov_colon)

# Observation
hybiss_colon = HybISS_Setup(tissue_colon)
hybiss_colon.observe_dots(fov_colon)

visualize_tissue(fov_colon, "Colon Crypts", tissue_colon, hybiss_colon)

---
## 2. Cerebral Cortex Layers

Simulating cortical tissue with 6 horizontal layers:
- Use ProbabilityNodeFieldRule to create smooth vertical gradients
- Different neuron types dominate in each layer

In [ ]:
# Cell type indices
# 0: L1 neurons, 1: L2/3 pyramidal, 2: L4 granular, 3: L5 pyramidal
# 4: L6 pyramidal, 5: PV interneurons, 6: Astrocytes

n_cell_types_cortex = 7
frame_size_cortex = 1000

# Create tissue expression profiles
tissue_cortex = TissueCellTypes()
tissue_cortex.generate_types_and_markers(n_genes=60, n_cell_types=n_cell_types_cortex)
tissue_cortex._cell_type_names = [
    "L1_Neuron", "L2/3_Pyramidal", "L4_Granular", "L5_Pyramidal",
    "L6_Pyramidal", "PV_Interneuron", "Astrocyte"
]

cell_props_cortex = CellTypesProperties(n_cell_types=n_cell_types_cortex)

# Define reference points for layered structure
# Y-axis represents depth (0=pial surface, frame_size=white matter)
reference_points = np.array([
    # Layer I (y ~ 50)
    [100, 50], [500, 50], [900, 50],
    # Layer II/III (y ~ 200)
    [100, 200], [500, 200], [900, 200],
    # Layer IV (y ~ 450)
    [100, 450], [500, 450], [900, 450],
    # Layer V (y ~ 650)
    [100, 650], [500, 650], [900, 650],
    # Layer VI (y ~ 850)
    [100, 850], [500, 850], [900, 850],
])

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Logits for each layer (will be converted to probabilities)
# Columns: L1, L2/3, L4, L5, L6, PV, Astro
logits = np.array([
    # Layer I - sparse, mostly L1 neurons and astrocytes
    [3, 0, 0, 0, 0, 0.5, 2], [3, 0, 0, 0, 0, 0.5, 2], [3, 0, 0, 0, 0, 0.5, 2],
    # Layer II/III - L2/3 pyramidal dominant + PV interneurons
    [0, 4, 0, 0, 0, 1.5, 0.5], [0, 4, 0, 0, 0, 1.5, 0.5], [0, 4, 0, 0, 0, 1.5, 0.5],
    # Layer IV - L4 granular dominant
    [0, 0.5, 4, 0, 0, 1, 0.5], [0, 0.5, 4, 0, 0, 1, 0.5], [0, 0.5, 4, 0, 0, 1, 0.5],
    # Layer V - L5 pyramidal dominant
    [0, 0, 0, 4, 0.5, 1, 0.5], [0, 0, 0, 4, 0.5, 1, 0.5], [0, 0, 0, 4, 0.5, 1, 0.5],
    # Layer VI - L6 pyramidal dominant
    [0, 0, 0, 0.5, 4, 1, 0.5], [0, 0, 0, 0.5, 4, 1, 0.5], [0, 0, 0, 0.5, 4, 1, 0.5],
])
ref_probs = softmax(logits)

cortex_rule = ProbabilityNodeFieldRule(
    n_cell_types=n_cell_types_cortex,
    n_ref_points=len(reference_points),
    ref_probs=ref_probs,
    reference_points=reference_points,
)

def cortex_background():
    return FrameWideElement(
        frame_size=frame_size_cortex,
        tipical_cell_spacing=18,
        rules=cortex_rule
    )

fov_dist_cortex = FOVDistribution(
    frame_size=frame_size_cortex,
    background_element=cortex_background,
)

fov_cortex = fov_dist_cortex.generate_fov()
cell_props_cortex.apply(fov_cortex)

hybiss_cortex = HybISS_Setup(tissue_cortex)
hybiss_cortex.observe_dots(fov_cortex)

visualize_tissue(fov_cortex, "Cerebral Cortex Layers", tissue_cortex, hybiss_cortex)

In [ ]:
# Show layer structure more clearly
fig, ax = plt.subplots(figsize=(12, 10))

# Color by Y position to verify layering
scatter = ax.scatter(
    fov_cortex.cell_centroids[:, 0],
    fov_cortex.cell_centroids[:, 1],
    c=fov_cortex.class_instance,
    cmap='tab10',
    s=20, alpha=0.7
)

# Add layer labels
layer_boundaries = [100, 300, 550, 750, 900]
layer_names = ['Layer I', 'Layer II/III', 'Layer IV', 'Layer V', 'Layer VI']
for i, (y, name) in enumerate(zip([50, 200, 450, 650, 875], layer_names)):
    ax.text(1050, y, name, fontsize=12, va='center')
    
# Draw layer boundaries
for y in layer_boundaries:
    ax.axhline(y, color='gray', linestyle='--', alpha=0.3)

ax.set_xlim(0, 1100)
ax.set_ylim(0, frame_size_cortex)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (depth - pial surface at top)')
ax.set_title('Cortical Layer Organization')
plt.colorbar(scatter, ax=ax, label='Cell Type')
plt.show()

---
## 3. Mammary Gland

Simulating breast tissue with:
- Stromal background (fibroblasts + adipocytes)
- Multiple acini with luminal-myoepithelial bilayer

In [ ]:
# Cell type indices
# 0: Luminal epithelial, 1: Myoepithelial, 2: Fibroblasts, 3: Adipocytes
# 4: Endothelial, 5: Immune

n_cell_types_breast = 6
frame_size_breast = 1200

tissue_breast = TissueCellTypes()
tissue_breast.generate_types_and_markers(n_genes=45, n_cell_types=n_cell_types_breast)
tissue_breast._cell_type_names = [
    "Luminal", "Myoepithelial", "Fibroblast", "Adipocyte", "Endothelial", "Immune"
]

cell_props_breast = CellTypesProperties(
    n_cell_types=n_cell_types_breast,
    sizes=[12, 10, 15, 25, 10, 8],  # Adipocytes are larger
    anisotropy=[0.85, 0.7, 0.8, 0.9, 0.8, 0.9],  # Myoepithelial more elongated
)

# Background: Stroma (fibroblasts + adipocytes)
def breast_background():
    return FrameWideElement(
        frame_size=frame_size_breast,
        tipical_cell_spacing=30,  # Sparse stroma
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_breast,
            list_N=[2, 3, 4, 5],  # Fibroblast, Adipocyte, Endothelial, Immune
            proportions=[0.35, 0.40, 0.15, 0.10]
        )
    )

# Foreground: Acini with luminal-myoepithelial bilayer
def breast_acinus():
    # Inner layer: Luminal (secretory)
    # Outer layer: Myoepithelial (contractile)
    rule = LayerRule(
        n_cell_types=n_cell_types_breast,
        layer_types=[0, 1],  # Luminal inner, Myoepithelial outer
        layer_boundaries=[0.65],  # 65% luminal, 35% myoepithelial
        transition_width=6,
    )
    return VacuolatedStructure(
        frame_size=frame_size_breast,
        scale=55 + np.random.uniform(-12, 12),
        hole_scale_factor=0.55,  # Lumen
        rules=rule,
        tipical_cell_spacing=10,  # Dense epithelium
    )

fov_dist_breast = FOVDistribution(
    frame_size=frame_size_breast,
    background_element=breast_background,
    other_elements=[breast_acinus],
    elements_frequency=[1.0],
    attempts_at_elements=12,  # ~12 acini
)

fov_breast = fov_dist_breast.generate_fov()
cell_props_breast.apply(fov_breast)

hybiss_breast = HybISS_Setup(tissue_breast)
hybiss_breast.observe_dots(fov_breast)

visualize_tissue(fov_breast, "Mammary Gland (Acini)", tissue_breast, hybiss_breast)

In [ ]:
# Zoom in on a single acinus to see the bilayer structure
fig, ax = plt.subplots(figsize=(10, 10))

# Find a region with acini (high density of luminal/myoepithelial)
luminal_mask = fov_breast.class_instance == 0
if np.sum(luminal_mask) > 0:
    center_x = np.median(fov_breast.cell_centroids[luminal_mask, 0])
    center_y = np.median(fov_breast.cell_centroids[luminal_mask, 1])
else:
    center_x, center_y = frame_size_breast/2, frame_size_breast/2

zoom_size = 250
x_min, x_max = center_x - zoom_size, center_x + zoom_size
y_min, y_max = center_y - zoom_size, center_y + zoom_size

mask = (
    (fov_breast.cell_centroids[:, 0] >= x_min) & 
    (fov_breast.cell_centroids[:, 0] <= x_max) &
    (fov_breast.cell_centroids[:, 1] >= y_min) & 
    (fov_breast.cell_centroids[:, 1] <= y_max)
)

# Draw cells as ellipses
ellipses = []
colors = []
for i in np.where(mask)[0]:
    ellipse = Ellipse(
        xy=(fov_breast.cell_centroids[i, 0], fov_breast.cell_centroids[i, 1]),
        width=2 * fov_breast.cell_major_axis[i],
        height=2 * fov_breast.cell_minor_axis[i],
        angle=np.degrees(fov_breast.cell_rotation[i]),
    )
    ellipses.append(ellipse)
    colors.append(fov_breast.cell_colors[i])

collection = PatchCollection(ellipses, alpha=0.7)
collection.set_facecolors(colors)
collection.set_edgecolors('black')
collection.set_linewidths(0.5)
ax.add_collection(collection)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('Acinus Detail: Luminal-Myoepithelial Bilayer')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=cell_props_breast.colordict[0], alpha=0.7, label='Luminal'),
    Patch(facecolor=cell_props_breast.colordict[1], alpha=0.7, label='Myoepithelial'),
    Patch(facecolor=cell_props_breast.colordict[2], alpha=0.7, label='Fibroblast'),
    Patch(facecolor=cell_props_breast.colordict[3], alpha=0.7, label='Adipocyte'),
]
ax.legend(handles=legend_elements, loc='upper right')

plt.show()

---
## Summary

This notebook demonstrated how to create tissue-specific simulations:

| Tissue | Key Features | Primary Rules Used |
|--------|--------------|--------------------|
| Colon Crypts | Multiple ring structures with stem cell gradients | `VacuolatedStructure` + `LayerRule` |
| Cortex | Horizontal layered organization | `ProbabilityNodeFieldRule` |
| Mammary Gland | Acini with luminal-myoepithelial bilayer | `VacuolatedStructure` + `LayerRule` |

The tissue descriptions in `tissue_descriptions/` folder provide detailed blueprints for implementing additional tissues.